In [10]:
from dataclasses import dataclass
import csv

In [11]:
class Block:
    def __init__(self, block_id, view):
        self.id = block_id
        self.view = view
    def __str__(self):
        return f"{{'id': '{self.id}', 'view': {self.view}}}"

@dataclass
class Vote:
    block_id: str
    def __hash__(self):
        return hash(self.block_id)

In [12]:
def read_input_csv(filename):
    inputs = []
    with open(filename, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t = row['type'].strip().lower()
            if t == 'block':
                bid = row['block_id'].strip()
                view = int(row['view'])
                inputs.append(Block(bid, view))
            elif t == 'vote':
                bid = row['block_id'].strip()
                inputs.append(Vote(bid))
    return inputs

In [13]:
class ChainBuilder:
    def __init__(self):
        self.chain = []
        self.votes = set()
        self.pending = {}

    def process_vote(self, vote):
        self.votes.add(vote.block_id)
        if vote.block_id in self.pending:
            self.try_add_block(self.pending[vote.block_id])

    def process_block(self, block):
        self.blocks[block.id] = block
        if block.id in self.votes and block.view == len(self.chain):
            self.try_add_block(block)
        else:
            self.pending[block.id] = block

    def try_add_block(self, block):
        if block.view == len(self.chain) and block.id in self.votes:
            self.chain.append(block)
            if block.id in self.pending:
                del self.pending[block.id]
            self.check_pending()
            return True
        return False

    def check_pending(self):
        added = True
        while added:
            added = False
            for bid in list(self.pending.keys()):
                blk = self.pending[bid]
                if blk.view == len(self.chain) and blk.id in self.votes:
                    self.chain.append(blk)
                    del self.pending[bid]
                    added = True
                    break

    def run(self, inputs):
        for item in inputs:
            if isinstance(item, Vote):
                self.process_vote(item)
            else:
                self.process_block(item)
        return self.chain

def print_chain(chain):
    print(f"\nConstructed chain:")
    for i, block in enumerate(chain):
        print(f"[{i}] {block} (genesis)")
    if chain:
        print(f"Current tip: view={chain[-1].view}, id={chain[-1].id}")
    else:
        print("Chain is empty")

In [15]:
inputs = read_input_csv('lab2/lab2.csv')

builder = ChainBuilder()
chain = builder.run(inputs)

print_chain(chain)


Constructed chain:
Chain is empty
